# BEATs (Job)
Deze notebook gebruikt de BEATs-implementatie uit `unilm/beats` voor de challenge.
De globale workflow is: model laden, optioneel TSE gebruiken, embeddings extraheren en visualiseren.
Voor je begint:
1. Clone `https://github.com/microsoft/unilm/tree/master/beats`.
2. Download `BEATs_iter3_plus_AS2M.pt` via de README van die repo.

In [8]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchaudio
from peft import LoraConfig, get_peft_model
from scipy.io import wavfile
from sklearn.decomposition import PCA

sys.path.append(
    r"C:\\Users\\20223669\\OneDrive - TU Eindhoven\\Documents\\GitHub\\Team-Internship-Sorama\\unilm\\beats"
)
from BEATs import BEATs, BEATsConfig

## Helper functions
Hier definiëren we hulpfuncties voor spectrogram-conversie, ruis toevoegen en audio-enhancement die later in training/extractie worden hergebruikt.

In [ ]:
STFT_N_FFT = 512
STFT_HOP_LENGTH = 160

# STFT settings used by helper functions and TSE preprocessing.

def waveform_to_spec(waveform):
    """Convert a waveform tensor to STFT magnitude."""
    spec = torch.stft(
        waveform,
        n_fft=STFT_N_FFT,
        hop_length=STFT_HOP_LENGTH,
        return_complex=True,
    )
    return torch.abs(spec)


def spec_to_waveform(spec, phase, length):
    """Reconstruct waveform from magnitude and phase tensors."""
    complex_spec = spec * torch.exp(1j * phase)
    waveform = torch.istft(
        complex_spec,
        n_fft=STFT_N_FFT,
        hop_length=STFT_HOP_LENGTH,
        length=length,
    )
    return waveform


def create_noisy_sample(waveform):
    """Add random Gaussian noise for denoising training pairs."""
    noise_level = torch.empty(1).uniform_(0.01, 0.05).item()
    noise = noise_level * torch.randn_like(waveform)
    return waveform + noise


def enhance_audio(waveform):
    """Enhance a waveform with the trained TSE model in the STFT domain."""
    spec = torch.stft(
        waveform,
        n_fft=STFT_N_FFT,
        hop_length=STFT_HOP_LENGTH,
        return_complex=True,
    )

    magnitude = torch.abs(spec)
    phase = torch.angle(spec)

    # Model expects [batch, channel, freq, time].
    magnitude = magnitude.unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        enhanced_mag = tse_model(magnitude).squeeze(0).squeeze(0)

    enhanced_wave = torch.istft(
        enhanced_mag * torch.exp(1j * phase),
        n_fft=STFT_N_FFT,
        hop_length=STFT_HOP_LENGTH,
        length=waveform.shape[-1],
    )

    return enhanced_wave

## Configuration block
In deze sectie zet je de belangrijkste experimentinstellingen (mode, sample rate, batch size, epochs en smoke-test opties).

In [ ]:
APPLY_AUGMENTATION = True
FEATURE_EXTRACTOR_MODE = "frozen"  # "frozen", "lora", "last_layer" or "adapter"
USE_TSE = True
TARGET_SAMPLE_RATE = 16000

BATCH_SIZE = 32
EPOCHS = 50

ADAPTER_DIM = 64  # Bottleneck size used when FEATURE_EXTRACTOR_MODE == "adapter"

QUICK_SMOKE_TEST = False  # True = faster TSE smoke run (subset + 1 epoch)
TSE_SMOKE_MAX_FILES = 128  # Used only when QUICK_SMOKE_TEST is True

## Load model
Hier laden we BEATs vanuit de geclonede repo en daarna de checkpoint-weights.
Als CUDA beschikbaar is, wordt automatisch de GPU gebruikt; anders CPU.
Na deze cel staat het basismodel klaar voor feature-extractie of fine-tuning.

In [12]:
checkpoint_path = r"C:\Users\20223669\OneDrive - TU Eindhoven\Documents\GitHub\Team-Internship-Sorama\BEATs_iter3_plus_AS2M.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu")

cfg = BEATsConfig(checkpoint["cfg"])
model = BEATs(cfg)
model.load_state_dict(checkpoint["model"])

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

## TSE model
Deze sectie bouwt en traint een eenvoudige speech/audio enhancement module die ruis kan onderdrukken voor robuustere features.

In [ ]:
# Optional: build a DataLoader for TSE training.
TSE_TRAIN_FOLDER = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\bearing\train"
TSE_SEGMENT_SECONDS = 1.0

quick_smoke_test = globals().get("QUICK_SMOKE_TEST", False)
smoke_max_files = globals().get("TSE_SMOKE_MAX_FILES", 128)
TSE_MAX_FILES = smoke_max_files if quick_smoke_test else None

if quick_smoke_test:
    print(f"Quick smoke mode enabled: limiting TSE loader to first {TSE_MAX_FILES} files.")

if "loader" in globals():
    del loader


def _load_waveform_for_tse(file_path, target_sr):
    """Load WAV from disk, convert to mono, and resample to target_sr."""
    try:
        waveform, sr = torchaudio.load(file_path)
    except RuntimeError:
        sr, data = wavfile.read(file_path)
        data = np.asarray(data)

        if np.issubdtype(data.dtype, np.integer):
            max_val = np.iinfo(data.dtype).max
            data = data.astype(np.float32) / float(max_val)
        else:
            data = data.astype(np.float32)

        if data.ndim == 1:
            waveform = torch.from_numpy(data).unsqueeze(0)
        else:
            waveform = torch.from_numpy(data.T)

        sr = int(sr)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0)
    else:
        waveform = waveform.squeeze(0)

    if sr != target_sr:
        waveform = torchaudio.functional.resample(
            waveform.unsqueeze(0),
            sr,
            target_sr,
        ).squeeze(0)

    return waveform


class TSEWaveformDataset(torch.utils.data.Dataset):
    """Dataset that returns fixed-length waveform segments as tuples."""

    def __init__(self, file_paths, target_sr, segment_seconds=1.0):
        self.file_paths = file_paths
        self.target_sr = target_sr
        self.segment_length = max(1, int(segment_seconds * target_sr))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        waveform = _load_waveform_for_tse(file_path, self.target_sr)

        # Random crop for long files; right-pad short files.
        if waveform.shape[0] >= self.segment_length:
            max_start = waveform.shape[0] - self.segment_length
            start = torch.randint(0, max_start + 1, (1,)).item() if max_start > 0 else 0
            waveform = waveform[start : start + self.segment_length]
        else:
            pad_len = self.segment_length - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))

        return (waveform,)


wav_paths = []
if os.path.isdir(TSE_TRAIN_FOLDER):
    for root, _, files in os.walk(TSE_TRAIN_FOLDER):
        for file_name in files:
            if file_name.lower().endswith(".wav"):
                wav_paths.append(os.path.join(root, file_name))
else:
    print(f"TSE_TRAIN_FOLDER does not exist: {TSE_TRAIN_FOLDER}")

wav_paths.sort()
if TSE_MAX_FILES is not None:
    wav_paths = wav_paths[:TSE_MAX_FILES]

if len(wav_paths) == 0:
    print("No WAV files found. Update TSE_TRAIN_FOLDER and rerun this cell.")
else:
    dataset = TSEWaveformDataset(
        wav_paths,
        target_sr=TARGET_SAMPLE_RATE,
        segment_seconds=TSE_SEGMENT_SECONDS,
    )
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
    )
    print(f"Created loader with {len(dataset)} files and batch size {BATCH_SIZE}.")

Created loader with 1000 files and batch size 32.


In [ ]:
class TSEModel(nn.Module):
    """Lightweight encoder-decoder for magnitude-spectrogram denoising."""

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(16, 8, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(8, 1, 2, stride=2, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        """Map noisy magnitude spectrograms to cleaned estimates."""
        z = self.encoder(x)
        out = self.decoder(z)
        return out


tse_model = TSEModel().to(device)

optimizer = torch.optim.Adam(tse_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

quick_smoke_test = globals().get("QUICK_SMOKE_TEST", False)
epochs = 1 if quick_smoke_test else 10

if quick_smoke_test:
    print("Quick smoke mode enabled: TSE training for 1 epoch.")

if not USE_TSE:
    print("USE_TSE is False, skipping TSE training.")
elif "loader" not in globals():
    print("No 'loader' found. Define a DataLoader named 'loader' before training TSE.")
else:
    for epoch in range(epochs):
        total_loss = 0.0

        for batch in loader:
            clean_wave = batch[0].to(device)
            noisy_wave = create_noisy_sample(clean_wave)

            # Supervise denoising in the magnitude-spectrogram domain.
            clean_spec = waveform_to_spec(clean_wave).unsqueeze(1).to(device)
            noisy_spec = waveform_to_spec(noisy_wave).unsqueeze(1).to(device)

            optimizer.zero_grad()
            pred = tse_model(noisy_spec)
            loss = loss_fn(pred, clean_spec)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print("Epoch", epoch, "Loss", total_loss)

Epoch 0 Loss 8.938174933195114
Epoch 1 Loss 3.298259101808071
Epoch 2 Loss 0.49689793633297086
Epoch 3 Loss 0.27211354579776525
Epoch 4 Loss 0.24402457755059004
Epoch 5 Loss 0.24359250720590353
Epoch 6 Loss 0.24132303101941943
Epoch 7 Loss 0.23834435269236565
Epoch 8 Loss 0.23934112396091223
Epoch 9 Loss 0.24125973461195827


## Preprocess data
Hier wordt audio ingelezen, eventueel geaugmenteerd en naar een consistente vorm gebracht vóór feature-extractie.
Dit zorgt dat korte clips, sample-rates en kanaalverschillen uniform behandeld worden.

In [ ]:
MIN_WAVEFORM_SAMPLES = 400


def augment_waveform(
    waveform,
    sample_rate,
    noise_prob=0.5,
    gain_prob=0.5,
    shift_prob=0.5,
    dropout_prob=0.3,
):
    """Apply simple time-domain augmentations to improve robustness."""
    if torch.rand(1).item() < noise_prob:
        noise_std = torch.empty(1).uniform_(0.001, 0.01).item()
        waveform = waveform + noise_std * torch.randn_like(waveform)

    if torch.rand(1).item() < gain_prob:
        gain = torch.empty(1).uniform_(0.8, 1.2).item()
        waveform = waveform * gain

    if torch.rand(1).item() < shift_prob:
        max_shift = int(0.1 * sample_rate)
        shift = torch.randint(-max_shift, max_shift + 1, (1,)).item()
        waveform = torch.roll(waveform, shifts=shift, dims=0)

    if torch.rand(1).item() < dropout_prob:
        drop_len = int(0.02 * sample_rate)
        start = torch.randint(0, max(1, waveform.shape[0] - drop_len), (1,)).item()
        waveform[start : start + drop_len] = 0

    waveform = torch.clamp(waveform, -1.0, 1.0)
    return waveform


def _load_wav_with_scipy(file_path):
    """Fallback WAV loader when torchaudio backend is unavailable."""
    sr, data = wavfile.read(file_path)
    data = np.asarray(data)

    if np.issubdtype(data.dtype, np.integer):
        max_val = np.iinfo(data.dtype).max
        data = data.astype(np.float32) / float(max_val)
    else:
        data = data.astype(np.float32)

    if data.ndim == 1:
        waveform = torch.from_numpy(data).unsqueeze(0)  # [1, time]
    else:
        waveform = torch.from_numpy(data.T)  # [channels, time]

    return waveform, int(sr)


def load_audio(file_path, target_sr=16000, apply_augment=False):
    """Load audio, enforce mono/target sample rate, and optionally augment."""
    try:
        waveform, sr = torchaudio.load(file_path)
    except RuntimeError:
        waveform, sr = _load_wav_with_scipy(file_path)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0)
    else:
        waveform = waveform.squeeze(0)

    if sr != target_sr:
        waveform = torchaudio.functional.resample(
            waveform.unsqueeze(0),
            sr,
            target_sr,
        ).squeeze(0)

    if waveform.shape[0] < MIN_WAVEFORM_SAMPLES:
        pad_len = MIN_WAVEFORM_SAMPLES - waveform.shape[0]
        waveform = torch.nn.functional.pad(waveform, (0, pad_len))

    if apply_augment:
        waveform = augment_waveform(waveform, target_sr)

    return waveform


def extract_beats_embedding(waveform):
    """Extract a single clip embedding by mean-pooling BEATs frame features."""
    waveform = waveform.to(device)
    with torch.no_grad():
        features, _ = model.extract_features(waveform.unsqueeze(0))  # [1, T, 768]
        clip_embedding = features.mean(dim=1).squeeze(0).cpu().numpy()  # [768]
    return clip_embedding

## Feature extractor training
In deze stap kies je hoe BEATs wordt gebruikt: volledig bevroren, laatste laag trainen, LoRA, of adapter-gebaseerde finetuning.

In [ ]:
if FEATURE_EXTRACTOR_MODE == "frozen":
    print("Using frozen BEATs")

    for param in model.parameters():
        param.requires_grad = False

elif FEATURE_EXTRACTOR_MODE == "last_layer":
    print("Fine-tuning last transformer layers")

    for param in model.parameters():
        param.requires_grad = False

    for param in model.encoder.layers[-1].parameters():
        param.requires_grad = True

elif FEATURE_EXTRACTOR_MODE == "lora":
    print("Using LoRA adaptation")

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

elif FEATURE_EXTRACTOR_MODE == "adapter":
    print("Using bottleneck adapters")

    for param in model.parameters():
        param.requires_grad = False

    class BottleneckAdapter(nn.Module):
        """Residual bottleneck adapter: down-project, activate, then up-project."""

        def __init__(self, hidden_dim, adapter_dim=64):
            super().__init__()
            self.down_proj = nn.Linear(hidden_dim, adapter_dim)
            self.act = nn.ReLU()
            self.up_proj = nn.Linear(adapter_dim, hidden_dim)

            # Start near-identity so adapters do not change features before training.
            nn.init.zeros_(self.up_proj.weight)
            nn.init.zeros_(self.up_proj.bias)

        def forward(self, x):
            return x + self.up_proj(self.act(self.down_proj(x)))

    class BEATsWithAdapters(nn.Module):
        """Wrapper that injects adapter layers into BEATs feature outputs."""

        def __init__(self, beats_model, hidden_dim, adapter_dim=64):
            super().__init__()
            self.beats_model = beats_model
            self.adapter = BottleneckAdapter(hidden_dim, adapter_dim=adapter_dim)

        def extract_features(self, *args, **kwargs):
            """Run BEATs feature extraction and adapt frame-level representations."""
            features, padding_mask = self.beats_model.extract_features(*args, **kwargs)
            features = self.adapter(features)
            return features, padding_mask

        def forward(self, *args, **kwargs):
            return self.beats_model(*args, **kwargs)

    adapter_dim = int(globals().get("ADAPTER_DIM", 64))
    hidden_dim = int(getattr(cfg, "encoder_embed_dim", 768))

    model = BEATsWithAdapters(
        model,
        hidden_dim=hidden_dim,
        adapter_dim=adapter_dim,
    ).to(device)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable_params}/{total_params}")

else:
    raise ValueError(
        "FEATURE_EXTRACTOR_MODE must be one of: 'frozen', 'last_layer', 'lora', 'adapter'."
    )

Using frozen BEATs


## Embeddings uit één machine halen (duurt ±10 min)
Het BEATs-model extraheert 768 features per clip. Deze embeddings kun je daarna in een classifier gebruiken.
Deze cel loopt door alle audiofiles en schrijft per clip een `.npy` embedding weg in dezelfde mapstructuur.

In [17]:
input_folder = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\bearing\train"
output_folder = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features"
os.makedirs(output_folder, exist_ok=True)

# Turn this off for validation/test feature extraction.
APPLY_AUGMENTATION = True

for root, _, files in os.walk(input_folder):
    for file_name in files:
        if not file_name.endswith(".wav"):
            continue

        file_path = os.path.join(root, file_name)
        waveform = load_audio(file_path, apply_augment=APPLY_AUGMENTATION)
        embedding = extract_beats_embedding(waveform)

        relative_path = os.path.relpath(root, input_folder)
        out_dir = os.path.join(output_folder, relative_path)
        os.makedirs(out_dir, exist_ok=True)

        out_path = os.path.join(out_dir, os.path.splitext(file_name)[0] + ".npy")
        np.save(out_path, embedding)

        print(f"Processed: {file_path} -> {out_path}")

print("All embeddings extracted successfully!")

Processed: C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\bearing\train\section_00_source_train_normal_0000_noAttribute.wav -> C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features\.\section_00_source_train_normal_0000_noAttribute.npy
Processed: C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\bearing\train\section_00_source_train_normal_0001_noAttribute.wav -> C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features\.\section_00_source_train_normal_0001_noAttribute.npy
Processed: C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\bearing\train\section_00_source_train_normal_0002_noAttribute.wav -> C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779\dev_bearing\features\.\section_00_source_train_normal_0002_noAttribute.npy
Processed: C:\Users\20223669\OneDrive

KeyboardInterrupt: 

## Visualize embeddings
Hier projecteren we embeddings met PCA naar 2D om snel te controleren of er structuur of clusters in de feature-ruimte zichtbaar zijn.

In [ ]:
def visualize_embeddings(embeddings):
    print("Running PCA visualization...")

    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(embeddings)

    plt.figure()
    plt.scatter(emb_2d[:, 0], emb_2d[:, 1], s=5)
    plt.title("Embedding space visualization (PCA)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()